# Lab 09: Kafka into Spark Structured Streaming

Kafka holds a topic called `lines`. In this notebook Spark reads it as a stream, counts words, and lands the raw lines in HDFS as Parquet.

Before you run it, create the topic and produce some lines (steps 2 and 3 of the lab README).

## 1. Start Spark
Spark runs inside this notebook container in local mode. Two shuffle partitions are plenty for a tiny stream.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("lab-09-kafka")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE = "hdfs://namenode:8020/user/lab/09"

## 2. Read the topic as a stream

`readStream` does not read anything yet. It describes a table that grows as messages arrive.
`startingOffsets=earliest` means "begin at the first message the topic still has".

In [ ]:
raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "lines")
    .option("startingOffsets", "earliest")
    .load()
)
raw.printSchema()

Kafka gives us bytes. Cast `value` to text to get the line we typed.

In [ ]:
lines = raw.select(F.col("value").cast("string").alias("line"), "timestamp")

## 3. Running word counts

`outputMode("complete")` rewrites the whole result table after each micro-batch. The `memory` sink keeps it as a temporary table called `word_counts` that we can query with SQL.

`processAllAvailable()` waits until Spark has read everything that is in the topic right now.

In [ ]:
words = lines.select(F.explode(F.split("line", r"\s+")).alias("word"))
counts = words.groupBy("word").count()

wc = (
    counts.writeStream.outputMode("complete")
    .format("memory").queryName("word_counts")
    .start()
)
wc.processAllAvailable()
spark.sql("SELECT * FROM word_counts ORDER BY count DESC, word LIMIT 8").show()

### Add more lines, then look again

Go back to your terminal, produce one or two more lines, then run the next cell. The counts grow without restarting the query.

In [ ]:
wc.processAllAvailable()
spark.sql("SELECT * FROM word_counts ORDER BY count DESC, word LIMIT 8").show()

In [ ]:
wc.stop()

## 4. Land the raw lines in HDFS as Parquet

Two things make this safe to re-run:

- **`checkpointLocation`**: Spark writes down which Kafka offsets each micro-batch covers. A new query with the same checkpoint continues where the last one stopped.
- **`trigger(availableNow=True)`**: process everything that is waiting, then stop by itself. Handy for notebooks and tests. A normal streaming job would use `processingTime="10 seconds"` and never stop.

In [ ]:
to_hdfs = (
    lines.writeStream.format("parquet")
    .option("path", f"{BASE}/lines")
    .option("checkpointLocation", f"{BASE}/checkpoints/lines")
    .trigger(availableNow=True)
    .start()
)
to_hdfs.awaitTermination()
print("batches in the last run:", len(to_hdfs.recentProgress))

Read the files back like any Parquet folder. Run the cell above again after producing more lines: the count goes up by exactly the new lines.

In [ ]:
saved = spark.read.parquet(f"{BASE}/lines")
print("rows in HDFS:", saved.count())
saved.orderBy("timestamp").show(truncate=False)

## 5. Peek inside the checkpoint

`offsets/N` is written before batch N runs: "I am about to read these offsets". `commits/N` is written after batch N is finished. The JSON on the last line of an offsets file maps topic to partition to the next offset to read.

In [ ]:
def hdfs_ls(path):
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    return sorted(s.getPath().getName() for s in fs.listStatus(p))

ckpt = f"{BASE}/checkpoints/lines"
for sub in hdfs_ls(ckpt):
    print(sub, hdfs_ls(f"{ckpt}/{sub}") if sub != "metadata" else "")

In [ ]:
spark.read.text(f"{ckpt}/offsets/0").show(truncate=False)